# Predictive Maintenance vs. Dynamic Scheduling Simulation
This notebook simulates a manufacturing asset under three maintenance paradigms:
1. **Run to Failure (RTF)**
2. **Preventive Maintenance (PM)**
3. **Predictive Maintenance (PdM)**

In [ ]:
import random
import statistics

class Machine:
    def __init__(self, failure_threshold=100.0):
        self.wear = 0.0
        self.failure_threshold = failure_threshold
        self.is_broken = False

    def operate(self):
        wear_increment = random.uniform(2.0, 5.0)
        self.wear += wear_increment
        if self.wear >= self.failure_threshold:
            self.is_broken = True
        return self.wear

    def read_sensor(self):
        sensor_noise = random.uniform(-1.5, 1.5)
        return max(0.0, self.wear + sensor_noise)

    def service(self):
        self.wear = 0.0
        self.is_broken = False

def simulate_strategy(strategy, total_cycles=1000, pm_interval=25, pdm_threshold=80.0):
    machine = Machine()
    total_profit = 0
    successful_cycles = 0
    planned_maint = 0
    unplanned_breakdowns = 0
    downtime_remaining = 0

    for cycle in range(1, total_cycles + 1):
        if downtime_remaining > 0:
            downtime_remaining -= 1
            if downtime_remaining == 0:
                machine.service()
            continue

        should_maint = False
        if strategy == "PM" and cycle % pm_interval == 0:
            should_maint = True
        elif strategy == "PdM" and machine.read_sensor() >= pdm_threshold:
            should_maint = True

        if should_maint:
            planned_maint += 1
            total_profit -= 300
            downtime_remaining = 2
            continue

        machine.operate()

        if machine.is_broken:
            unplanned_breakdowns += 1
            total_profit -= 1500
            downtime_remaining = 8
        else:
            successful_cycles += 1
            total_profit += 100

    return total_profit, successful_cycles, planned_maint, unplanned_breakdowns

random.seed(42)
runs = 100
strategies = ["RTF", "PM", "PdM"]
summary = {s: [] for s in strategies}

for _ in range(runs):
    for s in strategies:
        profit, _, _, _ = simulate_strategy(s)
        summary[s].append(profit)

print("=== MAINTENANCE STRATEGY SIMULATION RESULTS ===")
for s in strategies:
    mean_p = statistics.mean(summary[s])
    std_p = statistics.stdev(summary[s])
    print(f"Strategy: {s:<5} | Mean Net Profit: ${mean_p:,.2f} | Std Dev: ${std_p:,.2f}")